In [4]:
import random

# Parâmetros do Problema
C = [42.0, 35.0, 58.0, 30.0, 50.0, 65.0]  # Coeficientes de aquecimento das 6 AZs (°C)
CRITICAL_TEMP = 75.0                      # Limite crítico de temperatura (°C)

class Particle:
    def __init__(self, dim=6):
        self.dim = dim
        # Posição inicial aleatória (valores brutos antes da normalização)
        self.position = [random.uniform(0.1, 1.0) for _ in range(dim)]
        self.normalize_position()

        # Velocidade inicial
        self.velocity = [random.uniform(-0.1, 0.1) for _ in range(dim)]

        # Melhores pessoais (P_best)
        self.best_position = list(self.position)
        self.best_fitness = float('inf')

    def normalize_position(self):
        """Operador de normalização estrita: garante w_i >= 0 e sum(w_i) = 1.0"""
        self.position = [max(0.0, val) for val in self.position]
        total = sum(self.position)
        if total == 0:
            self.position = [1.0 / self.dim] * self.dim
        else:
            self.position = [val / total for val in self.position]

class ContinuousPSO:
    def __init__(self, num_particles, max_iter=100, w=0.5, c1=1.5, c2=1.5):
        self.num_particles = num_particles
        self.max_iter = max_iter
        self.w = w      # Inércia
        self.c1 = c1    # Coeficiente cognitivo
        self.c2 = c2    # Coeficiente social

        self.particles = [Particle() for _ in range(num_particles)]
        self.global_best_position = [0.0] * 6
        self.global_best_fitness = float('inf')
        self.fitness_history = []

    def evaluate_fitness(self, position):
        """Calcula a temperatura média ponderada e aplica penalidade se T > 75 °C"""
        temperatures = [C[i] * (1.0 + position[i] * 0.8) for i in range(6)]

        # Temperatura média ponderada do datacenter
        weighted_avg_temp = sum(position[i] * temperatures[i] for i in range(6))

        # Função de penalidade externa quadrática
        penalty = 0.0
        for t in temperatures:
            if t > CRITICAL_TEMP:
                penalty += 1000.0 * (t - CRITICAL_TEMP) ** 2

        return weighted_avg_temp + penalty

    def optimize(self):
        for _ in range(self.max_iter):
            for particle in self.particles:
                current_fitness = self.evaluate_fitness(particle.position)

                # Atualiza P_best
                if current_fitness < particle.best_fitness:
                    particle.best_fitness = current_fitness
                    particle.best_position = list(particle.position)

                # Atualiza G_best
                if current_fitness < self.global_best_fitness:
                    self.global_best_fitness = current_fitness
                    self.global_best_position = list(particle.position)

            # Atualiza velocidade e posição de cada partícula
            for particle in self.particles:
                for i in range(particle.dim):
                    r1 = random.random()
                    r2 = random.random()

                    cognitive = self.c1 * r1 * (particle.best_position[i] - particle.position[i])
                    social = self.c2 * r2 * (self.global_best_position[i] - particle.position[i])

                    particle.velocity[i] = self.w * particle.velocity[i] + cognitive + social
                    particle.position[i] += particle.velocity[i]

                # Aplica normalização obrigatória a cada iteração
                particle.normalize_position()

            self.fitness_history.append(self.global_best_fitness)

        return self.global_best_position, self.global_best_fitness, self.fitness_history

# Execução e Exibição no Colab
if __name__ == "__main__":
    pop_sizes = [10, 30, 50]
    results = {}

    print("="*80)
    print(f"{'POPULAÇÃO':<12} | {'MELHOR VETOR W':<45} | {'SOMA (sum w_i)':<15} | {'FITNESS'}")
    print("="*80)

    for pop in pop_sizes:
        pso = ContinuousPSO(num_particles=pop, max_iter=100)
        best_w, best_fit, history = pso.optimize()

        sum_w = sum(best_w)
        # Formata o vetor W com 4 casas decimais para exibição limpa
        formatted_w = "[" + ", ".join([f"{w:.4f}" for w in best_w]) + "]"

        print(f"{pop:<12} | {formatted_w:<45} | {sum_w:<15.6f} | {best_fit:.4f}")

    print("="*80)

POPULAÇÃO    | MELHOR VETOR W                                | SOMA (sum w_i)  | FITNESS
10           | [0.1633, 0.3209, 0.0000, 0.4786, 0.0372, 0.0000] | 1.000000        | 43.6402
30           | [0.1633, 0.3209, 0.0000, 0.4786, 0.0372, 0.0000] | 1.000000        | 43.6402
50           | [0.1633, 0.3209, 0.0000, 0.4786, 0.0372, 0.0000] | 1.000000        | 43.6402
